# 19장 · LCEL 체인과 출력 파서

LCEL의 파이프 연산자(|)는 앞 구성 요소의 출력을 다음 구성 요소의 입력으로 넘깁니다. 프롬프트, 모델, 출력 파서가 각각 주고받는 값이 다릅니다.

도서관 안내 문장과 질문을 LCEL 프롬프트에 전달합니다. 이어지는 모델과 출력 파서의 입력·반환 타입을 확인하고, 스트리밍 조각에서 전체 문자열을 복원합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langchain-core 1.6.12입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## LCEL 4시간과 SDK 문의 처리 4시간

앞의 네 시간에는 체인의 모델과 출력 파서를 바꿉니다. 뒤의 네 시간에는 아래 개인 실습 노트북에서 16~18일차 SDK 조회 흐름을 복원합니다.

1. 접수 ID가 있는 문의, ID가 없는 문의, 등록되지 않은 ID를 각각 넣으세요.
2. 정보 부족과 조회 결과 없음의 차이를 도구 호출 횟수와 응답으로 설명하세요.

[SDK 문의 처리 전용 노트북](https://langchain-lecture.vercel.app/chapter19/sdk-inquiry.ipynb)

## ChatPromptTemplate의 입력 변수

템플릿은 비어 있는 칸에 이번 자료를 넣어 요청을 만드는 틀입니다. context에는 근거를, question에는 묻는 문장을 넣습니다. 먼저 표준 파이썬으로 그 두 값을 확인합니다.

### 입력 키와 자료형 오류

<table><caption>증상과 확인</caption><thead><tr><th scope="col">증상</th><th scope="col">먼저 확인</th></tr>
</thead><tbody><tr><td>context 키 오류</td><td>템플릿의 변수 이름과 입력 키</td></tr>
<tr><td>AIMessage를 문자열처럼 처리</td><td>파서 연결 여부와 반환 타입</td></tr>
<tr><td>질문을 바꿔도 같은 답</td><td>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</td></tr>
<tr><td>모듈을 찾을 수 없음</td><td>현재 노트북 커널의 langchain-core 설치 여부</td></tr>
<tr><td>stream이 한꺼번에 보임</td><td>구성 요소의 스트리밍 지원·출력 처리</td></tr>
</tbody></table>

### 셀 01 · 프롬프트 입력 딕셔너리

context에는 근거 문장, question에는 사용자 질문을 전달합니다.

In [1]:
inputs = {"context": "수업용 가상 도서관의 대출 기간은 14일이다.", "question": "대출은 며칠인가요?"}
print(inputs["context"])

수업용 가상 도서관의 대출 기간은 14일이다.


프롬프트의 변수명 context는 입력 사전의 키와 일치해야 합니다.

여기서 질문만 ‘휴관일은 언제인가요?’로 고쳐 보세요. 질문 칸은 바뀌지만 context에는 여전히 대출 기간이 들어 있습니다. 템플릿에 값을 잘 채우는 것과 질문에 맞는 근거를 찾는 것은 서로 다른 작업입니다. 23장에서 질문에 맞는 문서를 찾고 24장에서 그 결과를 체인의 문맥으로 넘깁니다.

### 프롬프트 변수와 입력 키

<p>context는 모델에게 제공할 사실이고 question은 사용자의 질문입니다. 템플릿에 {context}와 {question}을 사용했으므로 invoke 입력에도 두 키가 있어야 합니다. 이름이 다르거나 빠지면 모델을 호출하기 전에 오류가 날 수 있습니다.</p>

<p>system 메시지에 넣는 자료도 자동으로 사실이 되는 것은 아닙니다. 이 예제는 명시된 가상 자료만 사용합니다. 이후 문서 검색 수업에서 context를 직접 작성하는 대신 검색한 문서로 만들 것입니다.</p>

### 과제 02 · context와 question 치환

파이썬 문자열 format에 inputs를 전달해 근거와 질문이 각각 들어간 문장을 출력하세요. 이 셀은 LCEL을 쓰기 전의 작은 연습입니다.

In [2]:
# {context}와 {question}을 실제 값으로 바꾸세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>template = &quot;근거: {context}\n질문: {question}&quot;
print(template.format(**inputs))</code></pre>
<p>실행 결과</p><pre><code>근거: 수업용 가상 도서관의 대출 기간은 14일이다.
질문: 대출은 며칠인가요?</code></pre>
<p>첫 줄에는 자료가, 둘째 줄에는 질문이 남습니다. 템플릿 자체를 화면에 찍으면 중괄호 안의 이름이 그대로 보입니다.</p>
</details>

#### 파이프로 연결할 때의 입력 형태

<mark>앞 단계의 반환값을 뒷단계가 받을 수 있어야 합니다.</mark> 템플릿에 필요한 질문과 근거를 채운 뒤 모델에 전달되는 문장을 읽어 보세요. 결과가 비어 있으면 파이프를 한꺼번에 따라가기보다 어느 경계에서 값의 형태가 바뀌었는지 살핍니다.

선수 개념 · 13장 순차 워크플로의 입력·출력 연결

## 파이프 연산자와 StrOutputParser, invoke()

입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.

prompt.invoke(inputs)는 메시지 묶음을 만듭니다. model과 파서를 연결하면 같은 입력의 최종 반환값은 문자열입니다.

### LCEL 구성 요소의 입출력

<p>Runnable은 invoke로 실행할 수 있는 구성 요소입니다. 먼저 프롬프트, 모델, 출력 파서를 연결하고, 뒤에서 문서 ID를 돌려주는 경로를 더합니다.</p>

<p>13장에서 다음 함수에 직접 건넸던 반환값을 여기서는 |가 다음 구성 요소로 전달합니다.</p>

<table><caption>연결 순서</caption><thead><tr><th scope="col">구성 요소</th><th scope="col">입력</th><th scope="col">출력</th></tr>
</thead><tbody><tr><td>ChatPromptTemplate</td><td>변수가 있는 딕셔너리</td><td>메시지를 담은 프롬프트 값</td></tr>
<tr><td>채팅 모델</td><td>메시지 프롬프트</td><td>AI 메시지</td></tr>
<tr><td>StrOutputParser</td><td>AI 메시지</td><td>문자열</td></tr>
</tbody></table>

### 셀 03 · to_messages()의 content

모델을 부르기 전에 ChatPromptTemplate이 만든 값에서 질문을 먼저 꺼내 봅니다.

In [ ]:
import os
from langchain_openai import ChatOpenAI

# 수업 커널에 %pip install langchain-openai 를 실행하고 OPENAI_API_KEY를 설정하세요.
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
prompt = ChatPromptTemplate.from_messages([("system", "주어진 사실만 사용하세요. 사실: {context}"),
                                           ("human", "{question}")])
print(prompt.invoke(inputs).to_messages()[-1].content)

마지막 메시지에는 ‘대출은 며칠인가요?’라는 질문이 들어 있습니다. prompt의 반환값 안에는 system 메시지도 들어 있습니다. 둘을 합친 메시지 묶음이 모델 입력입니다.

### 구성 요소별 중간 반환값

<p>ChatPromptValue는 메시지를 담은 프롬프트 값이며 to_messages()로 system과 human 메시지를 볼 수 있습니다. 채팅 모델의 AIMessage에는 content가 있습니다. StrOutputParser를 거친 뒤에는 문자열 자체를 사용합니다. 파서를 거친 값은 문자열처럼 쓸 수 있으므로 메시지 객체의 content를 다시 꺼낼 필요가 없습니다.</p>

<p>이 문자열 파서는 12일에 배운 JSON 스키마 검사나 내용 검증을 수행하지 않습니다. 18일의 입력 검사 책임도 사라지지 않습니다. 모델이 돌려준 문장이 사실인지 확인하려면 별도 근거와 평가가 필요합니다.</p>

### 셀 04 · StrOutputParser와 invoke()

입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.

모델 뒤에서 멈추면 답은 메시지 객체에 담겨 있습니다. 뒤에 StrOutputParser를 연결하면 invoke의 반환값으로 본문 문자열을 얻습니다. 이 노트북의 실행 기록은 langchain-core 1.6.12 기준이고 더 읽기의 1.8.0은 별도 기록입니다.

In [ ]:
chain = prompt | model | StrOutputParser()
print(chain.invoke(inputs))

대출 기간은 14일입니다.


입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.

### 실제 모델 응답 확인

<p>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</p>

<p>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</p>

## stream()과 출력 조각 결합

invoke는 한 번에 결과를 받고 stream은 차례로 오는 조각을 읽습니다. 조각 하나의 경계는 문장이나 단어의 경계와 같지 않을 수 있습니다.

### 과제 05 · 스트림 문자열 결합

LCEL 환경에서는 chain.stream의 조각을 이어 전체 답을 출력하세요. 라이브러리가 없는 경우 제공 조각으로 결합 연산만 연습합니다.

In [ ]:
# chain.stream(inputs)로 받은 조각을 순서대로 이어 보세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>pieces = chain.stream(inputs)
print(&quot;&quot;.join(pieces))</code></pre>
<p>실행 결과</p><pre><code>가상 도서관의 대출 기간은 14일입니다.</code></pre>
<p>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</p>
</details>

### 조각을 도착 순서대로 이어 붙이기

<p>pieces에는 ‘가상 ’과 ‘안내’ 두 문자열이 들어 있습니다. 빈 문자열로 join하면 첫 조각 끝에 들어 있던 공백을 보존한 ‘가상 안내’가 됩니다. 이 셀은 조각을 다시 합치는 연습으로, 실제 모델이 언제 어떤 길이로 보냈는지는 측정하지 않습니다.</p>

## 돌아보기

입력 사전, 모델에 보낼 메시지 묶음, 마지막 문자열을 순서대로 가리켜 보세요. 다음 장에서는 context를 손으로 쓰는 대신 어느 문서에서 가져올지 정합니다.

### 과제 06 · 프롬프트가 받은 문맥 확인

질문은 유지하고 대출 규정만 7일로 바꿉니다. 채팅 모델에 넘겨질 실제 메시지를 출력한 다음 체인의 답도 확인하세요. langchain-core를 설치한 커널에서 실행합니다.

In [ ]:
revised = {**inputs, "context": "이번 달 대출 기간은 7일입니다."}
# 시스템 메시지에서 revised의 문장이 보이는지 확인하고 체인의 답을 출력하세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>revised = {**inputs, &quot;context&quot;: &quot;이번 달 대출 기간은 7일입니다.&quot;}

messages = prompt.invoke(revised).to_messages()
print(messages[0].content)
print(chain.invoke(revised))</code></pre>
<p>실행 결과</p><pre><code>주어진 사실만 사용하세요. 사실: 이번 달 대출 기간은 7일입니다.
가상 도서관의 대출 기간은 14일입니다.</code></pre>
<p>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</p>
</details>

## 자료와 출처를 함께 돌려주는 LCEL 체인 만들기

기존 체인은 모델 답변 문자열 한 개를 돌려줬습니다. 이제 화면에서 받은 문의는 query, 근거 안내문은 notice 안에 들어옵니다. 입력을 기존 체인에 맞게 바꾼 뒤, 화면으로 돌아갈 때 문서 ID도 보존해 보세요. langchain-core가 설치된 수업 커널을 사용합니다.

### 셀 08 · 화면 입력을 프롬프트가 받을 값으로 바꾸기

RunnableLambda로 화면 입력을 context와 question에 대응시킵니다. 모델을 연결하기 전에 시스템·사용자 메시지 중 사용자 메시지를 꺼내 확인하세요.

In [6]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

ticket = {"query": "대출은 며칠인가요?",
          "notice": {"id": "guide-1", "text": inputs["context"]}}
def as_prompt_inputs(ticket):
    return {"context": ticket["notice"]["text"], "question": ticket["query"]}

adapt_input = RunnableLambda(as_prompt_inputs)
ready_prompt = (adapt_input | prompt).invoke(ticket)
print(ready_prompt.to_messages()[-1].content)

대출은 며칠인가요?


화면은 query라는 키를 보냈고 프롬프트는 question을 읽었습니다. 두 이름을 연결한 줄을 지나 실제 질문이 사용자 메시지에 들어갔습니다.

### 문의 한 건에서 답변과 출처가 돌아오는 두 경로

```mermaid
flowchart TD
    ticket["화면 입력 ticket"] --> adapt["context와 question 꺼내기"]
    adapt --> chain["기존 체인으로 답변 생성"]
    ticket --> lookup["안내문의 id 읽기"]
    chain --> collect["answer와 source 반환"]
    lookup --> collect
```

RunnableParallel은 두 갈래에 같은 ticket을 줍니다. answer는 기존 체인이 돌려준 문자열이고 source는 원래 안내문의 ID입니다. 도식은 값이 전달되는 경로를 표시하며 어느 쪽이 먼저 끝나는지는 정하지 않습니다.

### 과제 09 · 한 입력에서 답변과 문서 ID를 함께 반환하기

RunnableParallel의 answer에는 adapt_input | chain을, source에는 원래 ticket에서 notice.id를 꺼내는 RunnableLambda를 연결하세요. 완성한 체인을 한 번 호출해 출처와 답변을 출력하세요.

In [7]:
# 같은 ticket을 받는 answer와 source 두 갈래를 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>source_from_ticket = RunnableLambda(lambda item: item[&quot;notice&quot;][&quot;id&quot;])
with_source = RunnableParallel(answer=adapt_input | chain, source=source_from_ticket)
result_with_source = with_source.invoke(ticket)
print(result_with_source[&quot;source&quot;])
print(result_with_source[&quot;answer&quot;])</code></pre>
<p>실행 결과</p><pre><code>guide-1
가상 도서관의 대출 기간은 14일입니다.</code></pre>
<p>입력의 대출 기간을 14일에서 7일로 바꾸세요. 시스템 메시지의 숫자와 생성한 답변의 숫자를 비교하면 입력 전달과 생성 오류를 구분할 수 있습니다.</p>
</details>

<mark>출처를 모델 답변 뒤에서 복원할 필요가 없도록 원래 입력에서 따로 전달하세요.</mark> 답변 생성에 쓰는 프롬프트에는 문서 본문이 필요하고, 화면의 근거 링크에는 문서 ID가 필요합니다. 두 갈래가 같은 입력을 받으면 어느 단계가 어느 값을 사용했는지 확인할 수 있습니다.

문서 로더가 한 문서에서 만든 본문과 메타데이터를 나중에 같은 방식으로 분리해 전달할 수 있습니다.